# SpecMod tutorial

Fitting a source model to the spectra of a small induced earthquake, end to end.

The event is the Preston New Road **Mw 1.6** of 2019-08-26, recorded on the
LV and UR networks during hydraulic fracturing near Blackpool, UK. The
waveforms and station metadata are committed with the package (224 KB), so
this notebook needs no network access.

Five stages, one section each:

1. **Read and prepare** the waveforms — geometry, picks, instrument response.
2. **Cut** a signal window and a noise window to judge it against.
3. **Transform** both to spectra and select the usable bandwidth.
4. **Fit** a source model over that bandwidth.
5. **Save** the results.

Every processing step is written down with its equation in
[`docs/processing.md`](../docs/processing.md); this is the same pipeline with
the numbers left in.

## 1. Read and prepare

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
from obspy import UTCDateTime, read, read_inventory

import specmod.preprocess as pre
import specmod.utils as ut

# Relative to this notebook, so it runs from wherever it is opened.
DATA = Path("Data/2019-08-26T07:30:47.0")
METADATA = Path("MetaData")
OUTPUT = Path("Spectra")

The origin is what distances and theoretical arrivals are measured from.

Note the origin time is 18 minutes *after* the waveform start — the records
here begin at 07:30:47 and the catalogue origin is 07:49:24. Nothing in this
notebook depends on that offset, because every window is cut relative to the
**picks** rather than to the origin. It matters in one place only, and
`preprocess` now warns rather than producing nonsense: see
`set_picks_from_pyrocko`, which will not extrapolate a missing S arrival when
the origin does not precede the P pick.

In [2]:
olat, olon, odep = 53.784, -2.967, 2.1
otime = UTCDateTime("2019-08-26T07:49:24.2")

In [3]:
# The two horizontal components. S-wave amplitudes are what the source model
# is fitted to, so the verticals are not read.
st = read(str(DATA / "*HHE*"), format="mseed") + read(
    str(DATA / "*HHN*"), format="mseed"
)
inv = read_inventory(str(METADATA / "pnr_inventory.xml"), "stationxml")
print(f"{len(st)} traces, {len({tr.stats.station for tr in st})} stations")

28 traces, 14 stations


In [4]:
# Source-receiver geometry, from the inventory. Sets repi, rhyp, azimuth and
# back-azimuth on every trace.
pre.set_stream_distance(st, olat, olon, odep, otime, inventory=inv, dtype="mseed")

# P and S arrivals, from a Snuffler marker file.
pre.set_picks_from_pyrocko(st, str(next(DATA.glob("*.picks"))))

# A trace with no S pick cannot have an S window cut from it. Dropping them
# here is the pipeline's own idiom for "unusable".
for tr in st:
    if "s_time" not in tr.stats:
        st.remove(tr)

print(f"{len(st)} traces with both picks")

28 traces with both picks


/home/user/SpecMod/.venv/lib/python3.11/site-packages/obspy/core/inventory/network.py:321: UserWarning: Found more than one matching channel metadata. Returning first.
  warnings.warn(msg)


**Instrument correction happens here, in the notebook, not inside the
package.** That is deliberate: the choice of output units and water level is
part of the science, and burying it in a library call makes it invisible in
the record of what was run.

Demeaning matters more than it looks. A non-zero mean puts all of its energy
in the DC bin, which every estimator discards — so leaving it in loses energy
that the Parseval check would report as a failure.

In [5]:
st.detrend("linear")
st.detrend("demean")
st.taper(0.05)
st.remove_response(inv, output="VEL")   # ground velocity, m/s

/home/user/SpecMod/.venv/lib/python3.11/site-packages/obspy/core/inventory/network.py:251: UserWarning: Found more than one matching response. Returning first.
  warnings.warn(msg)


28 Trace(s) in Stream:

LV.L001..HHE | 2019-08-26T07:30:46.900000Z - 2019-08-26T07:30:53.900000Z | 100.0 Hz, 701 samples
...
(26 other traces)
...
UR.AQ10.00.HHN | 2019-08-26T07:30:49.500000Z - 2019-08-26T07:30:56.500000Z | 100.0 Hz, 701 samples

[Use "print(Stream.__str__(extended=True))" to print all Traces]

In [6]:
ut.plot_traces(st.copy(), plot_theoreticals=True, conv=1)
plt.show()

## 2. Cut the windows

The S-window opens at a fixed fraction of the elapsed P–S time after the P
arrival, then is **refined** onto the part of it that actually carries energy:
the window is tightened to where the cumulative squared amplitude runs between
its 1st and 99th percentiles. That is why these come out at 1.8–3.7 s rather
than the nominal 20 s.

The noise window ends shortly before the P arrival and is *asked for* the same
length as the refined signal window. It rarely gets it — see below.

In [7]:
sig = pre.get_signal(
    st, pre.cut_s, rafp=0.8, tafs=20, time_after="absolute_time", refine_window=True
)
noise = pre.get_noise_p(st, sig)

In [8]:
# Every noise window here is shorter than the signal it is judged against,
# because the records begin only ~2 s before the P arrival. This is normal and
# is corrected for; it is why `SpectrumPair` carries a resolution floor rather
# than assuming the two spectra share a frequency resolution.
for s, n in list(zip(sig, noise))[:4]:
    asked = float(n.stats["wend_requested"] - n.stats["wstart_requested"])
    got = float(n.stats["wend"] - n.stats["wstart"])
    print(
        f"{s.id:16s} signal {s.stats.endtime - s.stats.starttime:5.2f} s   "
        f"noise asked {asked:5.2f} s, got {got:5.2f} s"
    )

LV.L001..HHE     signal  3.04 s   noise asked  3.04 s, got  1.63 s
LV.L002..HHE     signal  3.34 s   noise asked  3.34 s, got  1.51 s
LV.L006..HHE     signal  2.90 s   noise asked  2.90 s, got  1.42 s
LV.L007..HHE     signal  2.79 s   noise asked  2.79 s, got  1.48 s


In [9]:
ut.plot_traces(st.copy(), plot_windows=True, conv=1, sig=sig, noise=noise)
plt.show()

## 3. Spectra and bandwidth

`spectrum_set_from_streams` transforms both windows, puts the noise on the
signal's frequency axis, bins both, raises the noise to account for what sits
*under* the signal, takes the ratio and selects the band where it passes.

Which estimator is used is configuration, not code: `fft`, `welch`,
`multitaper`, `quadratic` and `cwt` all satisfy the same Parseval contract, so
they are interchangeable here. The shipped default is `multitaper`.

In [10]:
from specmod.pipeline import spectrum_set_from_streams

spectra = spectrum_set_from_streams(sig, noise)
print(f"{len(spectra)} spectra for event {spectra.event}")
print(f"{sum(p.passes for p in spectra.pairs.values())} passed the signal-to-noise gate")

28 spectra for event 2019-08-26T07:49:24.200000Z
28 passed the signal-to-noise gate


In [11]:
# One station in detail: signal, noise, the binned spectra the ratio is
# actually computed on, the selected band (red) and the resolution floor (grey).
from specmod.plotting import plot_pair, plot_set

station = spectra.ids()[0]
plot_pair(spectra[station], id=station, show_binned=True)
plt.show()

In [12]:
pair = spectra[station]
print(f"band            {pair.band[0]:.2f} to {pair.band[1]:.2f} Hz")
print(f"resolution floor {pair.resolution_floor:.2f} Hz  (the shorter window's 1/T)")
print(f"units            {pair.signal.unit}")

band            0.65 to 47.39 Hz
resolution floor 0.61 Hz  (the shorter window's 1/T)
units            m/s*s


### Changing ground-motion domain

The response was removed to velocity, but a source model is often read on
displacement. `to_motion` converts the whole event and **returns a new set** —
the velocity one is untouched, so both exist at once.

In [13]:
displacement = spectra.to_motion("displacement")
print(f"velocity     {spectra[station].signal.unit}")
print(f"displacement {displacement[station].signal.unit}")

# The unbinned signal-to-noise ratio is invariant under this — both spectra are
# divided by the same 2*pi*f — but the *binned* ratio is not, because a bin
# holds a geometric mean. So a few bands do move.
moved = [i for i in spectra.ids() if spectra[i].band != displacement[i].band]
print(f"bands that moved: {len(moved)} of {len(spectra)}")

velocity     m/s*s
displacement m*s
bands that moved: 0 of 28


## 4. Fit a source model

The model comes from configuration — a Brune source with constant Q by
default — and the initial guesses are derived from each spectrum: the plateau
from the largest amplitude inside the band, the corner from where that
maximum falls.

`FitSpectra(spectra)` then `fit_spectra()` is the whole thing. The minimiser
(Powell), the `t*` lower bound and whether to fit the binned or unbinned
spectrum all come from `[fitting]` in the configuration, so the defaults are
recorded rather than remembered.

In [14]:
from specmod import sources
from specmod.fitting import FitSpectra, initial_guess

print(sources.from_config().describe())
guess = initial_guess(spectra)
print(f"guesses for {len(guess)} of {len(spectra)} stations")

brune+constant_q in velocity
guesses for 28 of 28 stations


In [15]:
fits = FitSpectra(spectra)
fits.fit_spectra()
print(f"{len(fits.models)} fitted, {fits.table['pass_fitting'].sum()} passed the fit checks")

28 fitted, 28 passed the fit checks


The guess is only a starting point, and a crude one: it takes the largest
amplitude inside the band, which on a velocity spectrum is *near* the corner
but can land at the band edge when the corner sits outside the resolvable
range. Comparing it with where the fit ended up shows how much work the
minimiser is doing.

In [16]:
import pandas as pd

fitted = fits.table.set_index("id")["fc"]
comparison = pd.DataFrame(
    {
        "guessed fc": {k: v["fc"] for k, v in guess.items()},
        "fitted fc": fitted,
        "band high": {k: spectra[k].band[1] for k in guess},
    }
).round(2)
comparison.head(8)

,guessed fc,fitted fc,band high
LV.L001..HHE,47.21,4.39,47.39
LV.L002..HHE,4.78,5.89,33.67
LV.L006..HHE,3.44,3.01,38.85
LV.L007..HHE,3.57,6.37,26.30
LV.L008..HHE,2.92,19.42,18.89
LV.L009..HHE,3.10,4.32,45.74
UR.AQ01.00.HHE,5.14,9.86,30.85
UR.AQ03.00.HHE,5.92,2.83,77.70


In [17]:
# The fitted model over the spectrum it was fitted to.
plot_pair(spectra[station], id=station, fit=fits.models[station])
plt.show()

In [18]:
# Or the whole event at once.
plot_set(spectra, fits=fits, columns=4)
plt.show()

In [19]:
fits.table[["id", "fc", "fc-stderr", "llpsp", "ts", "pass_fitting"]].head(10)

,id,fc,fc-stderr,llpsp,ts,pass_fitting
0,LV.L001..HHE,4.394520,None,-4.614151,0.023064,True
1,LV.L002..HHE,5.888485,None,-5.154718,0.041821,True
2,LV.L006..HHE,3.007301,None,-4.601092,0.066537,True
3,LV.L007..HHE,6.368972,None,-5.546449,0.061823,True
4,LV.L008..HHE,19.418044,None,-5.793181,0.065783,True
5,LV.L009..HHE,4.322822,None,-5.045543,0.032677,True
6,UR.AQ01.00.HHE,9.862871,None,-5.573028,0.079704,True
7,UR.AQ03.00.HHE,2.834860,None,-4.749034,0.019050,True
8,UR.AQ04.00.HHE,6.356409,None,-5.009011,0.012962,True
9,UR.AQ05.00.HHE,59.638328,None,-4.759449,0.046850,True


### The fit is not unique, and that is not a detail

`fc-stderr` is empty above. Powell — the shipped default, and what the
published workflow used — searches without building a covariance matrix, so
lmfit has no uncertainties to report. The obvious response is to use a method
that does:

```python
fits.fit_spectra(method="leastsq")
```

But the two do not merely differ in whether they report an error bar. **They
land on different answers**, and comparing them is the most useful thing this
notebook can show you, because it makes visible something a single fit hides:
minimising misfit does not have one solution here, and choosing between the
candidates is your job rather than the optimiser's.

In [20]:
alternatives = {}
for method in ("powell", "leastsq"):
    run = FitSpectra(spectra)
    run.fit_spectra(method=method)
    alternatives[method] = run

comparison = pd.DataFrame(
    {
        "fc powell": alternatives["powell"].table.set_index("id")["fc"],
        "fc leastsq": alternatives["leastsq"].table.set_index("id")["fc"],
        "redchi powell": alternatives["powell"].table.set_index("id")["redchi"],
        "redchi leastsq": alternatives["leastsq"].table.set_index("id")["redchi"],
    }
)
comparison["fc ratio"] = comparison["fc leastsq"] / comparison["fc powell"]

# Ordered by how much the two disagree.
comparison.reindex(
    (comparison["fc ratio"] - 1).abs().sort_values(ascending=False).index
).head(6).round(3)

,fc powell,fc leastsq,redchi powell,redchi leastsq,fc ratio
id,,,,,
UR.AQ10.00.HHN,21.256,14.747,0.026,0.025,0.694
UR.AQ03.00.HHN,2.966,2.737,0.127,0.127,0.923
LV.L001..HHN,2.983,3.122,0.344,0.344,1.047
LV.L008..HHE,19.418,19.201,0.028,0.028,0.989
UR.AQ06.00.HHE,6.271,6.308,0.048,0.048,1.006
LV.L008..HHN,23.157,23.063,0.032,0.032,0.996


Most stations agree to a fraction of a percent. A few do not, and the top of
that table is the interesting part: the two minimisers reach **the same
reduced chi-squared to within a few percent** at corner frequencies that
differ by tens of percent.

That matters more than the number suggests. Stress drop scales as $f_c^3$, so
a corner frequency ratio of 1.4 is a factor of **three** in stress drop —
between two fits you cannot tell apart on goodness of fit alone.

In [21]:
worst = (comparison["fc ratio"] - 1).abs().idxmax()
row = comparison.loc[worst]
ratio = row["fc powell"] / row["fc leastsq"]
print(f"{worst}")
for method in ("powell", "leastsq"):
    print(f"  {method:8s} fc = {row['fc ' + method]:6.2f} Hz"
          f"   reduced chi-sq {row['redchi ' + method]:.4f}")
print(f"  ratio in fc         {ratio:.2f}")
print(f"  implied stress-drop ratio (fc^3)  {ratio**3:.2f}")

UR.AQ10.00.HHN
  powell   fc =  21.26 Hz   reduced chi-sq 0.0259
  leastsq  fc =  14.75 Hz   reduced chi-sq 0.0254
  ratio in fc         1.44
  implied stress-drop ratio (fc^3)  2.99


In [22]:
# Both models over the spectrum they were fitted to.
plot_pair(
    spectra[worst],
    id=worst,
    fit={m: run.models[worst] for m, run in alternatives.items()},
)
plt.show()

Look at where they differ: high up the falling limb, where the source corner
and the attenuation $t^*$ trade off against each other. Both curves pass
through the data; they disagree about how much of the high-frequency falloff
is the *source* rolling off and how much is the *path* absorbing. That
trade-off is intrinsic to fitting $\Omega$, $f_c$ and $t^*$ together from one
spectrum, and no minimiser can resolve it — it needs either more information
(several stations, a spectral ratio, an independent $Q$) or a decision.

So: fit both ways, look at the spread, and record which you chose.
`[fitting] method` in the configuration is where that choice belongs, so that
a run says what it did rather than relying on anyone's memory.

`pass_fitting` marks a fit whose parameter is pinned against one of its
bounds — the minimiser saying "further, if you would let me", with the bound
reported instead of a measurement. Without uncertainties the test is weaker,
because it can only ask whether the value *is* the bound rather than whether
it reaches one.

In [23]:
for method, run in alternatives.items():
    table = run.table
    print(f"{method:9s} {table['pass_fitting'].sum():2d}/{len(table)} passed"
          f"   uncertainties reported: {table['fc-stderr'].notna().sum()}/{len(table)}")

powell    28/28 passed   uncertainties reported: 0/28
leastsq   22/28 passed   uncertainties reported: 28/28


### Why the answer comes from many stations, and from two stages

The ambiguity above is a property of **one spectrum**, and that is the whole
reason the method does not use one. $f_c$ belongs to the *source* — every
station is looking at the same rupture, so there is one value of it for the
event. $t^*$ belongs to the *path*, and every station has a different one.

So the trade-off that makes a single fit ambiguous does not point the same way
twice. A station whose $t^*$ is over-estimated returns a corner that is too
high; one whose $t^*$ is under-estimated returns a corner that is too low.
Averaging over the ensemble is not a cosmetic smoothing — it is using the fact
that the quantity being averaged is common to all of them while the
contaminating one is not.

**Stage 1** is what has been run so far: $\Omega$, $f_c$ and $t^*$ free at
every station independently. Its output is not the answer; it is 28 noisy
estimates of one number, plus 28 estimates of 28 different numbers.

In [24]:
# Weighted by inverse distance: the nearer station has less path between the
# source and the sensor, so less of its high-frequency falloff can be
# attenuation, and its corner is the better constrained of the two.
#
# `repi` — epicentral — because that is what `[windows] distance_metric` says.
# Which distance you use is not a detail at short range: here the nearest
# station is 0.89 km epicentral against 2.30 km hypocentral. `rhyp` is built
# from the source depth and the station *elevation*, so it assumes every
# sensor is at the surface; where sensor depths are unknown, as they are here,
# epicentral is the honest choice.
metric = "repi"
weight = pd.Series(
    {id: 1.0 / spectra[id].signal.meta[metric] for id in fits.models}
)

event_fc = {}
for method, run in alternatives.items():
    fc = run.table.set_index("id")["fc"]
    w = weight.reindex(fc.index)
    event_fc[method] = float((fc * w).sum() / w.sum())
    print(f"{method:9s} event fc = {event_fc[method]:6.3f} Hz")

worst_ratio = row["fc powell"] / row["fc leastsq"]
event_ratio = event_fc["powell"] / event_fc["leastsq"]
print()
print(f"worst single station:  fc ratio {worst_ratio:.3f}"
      f"   -> stress drop {worst_ratio**3:.2f}x")
print(f"event ensemble:        fc ratio {event_ratio:.3f}"
      f"   -> stress drop {event_ratio**3:.3f}x")

powell    event fc = 11.585 Hz
leastsq   event fc = 11.535 Hz

worst single station:  fc ratio 1.441   -> stress drop 2.99x
event ensemble:        fc ratio 1.004   -> stress drop 1.013x


A factor of three in stress drop at the worst station becomes **under 2%**
across the event. The choice of minimiser stopped mattering — not because one
of them was right, but because what they disagreed about was not the source.

**Stage 2** then fixes $f_c$ at that event value and refits, so each station
solves for its own $\Omega$ and $t^*$ against a corner frequency it is no
longer allowed to trade against. `set_const` is the whole of it.

In [25]:
stage_two = {}
for method in alternatives:
    run = FitSpectra(spectra)
    run.set_const("fc", event_fc[method])
    run.fit_spectra(method=method)
    stage_two[method] = run

second = pd.DataFrame(
    {
        f"{name} {method}": run.table.set_index("id")[column]
        for column, name in (("ts", "t*"), ("llpsp", "log10 omega"))
        for method, run in stage_two.items()
    }
)
t_star = 100 * (second["t* powell"] / second["t* leastsq"] - 1).abs().max()
omega = (second["log10 omega powell"] - second["log10 omega leastsq"]).abs().max()
print(f"t*     worst disagreement between minimisers: {t_star:.2f}%")
print(f"omega  worst disagreement between minimisers: {omega:.1e} log10 units")
second.head(6).round(4)

t*     worst disagreement between minimisers: 0.23%
omega  worst disagreement between minimisers: 1.8e-03 log10 units


,t* powell,t* leastsq,log10 omega powell,log10 omega leastsq
id,,,,
LV.L001..HHE,0.0326,0.0326,-4.9657,-4.9649
LV.L002..HHE,0.0529,0.0529,-5.2984,-5.2979
LV.L006..HHE,0.0820,0.0819,-5.1084,-5.1078
LV.L007..HHE,0.0752,0.0751,-5.6139,-5.6138
LV.L008..HHE,0.0533,0.0532,-5.8285,-5.8286
LV.L009..HHE,0.0427,0.0427,-5.4009,-5.4001


0.3% in $t^*$ and about 0.002 in $\log_{10}\Omega$ — which is 0.003 magnitude
units, against the 0.2 that folding the spectrum the wrong way would cost you.

Be clear about what that agreement is and is not. It is **not** evidence that
stage 2 resolved the trade-off: fixing $f_c$ removes the parameter the two
minimisers were disagreeing about, so of course they now agree. What it does
show is that the remaining two-parameter problem is well conditioned — once
the corner is pinned, $\Omega$ and $t^*$ are *determined* by the spectrum
rather than negotiable. The judgement is concentrated into one number for the
whole event, and that number came from the ensemble rather than from a
minimiser's preference.

This is why the published Magna and PNR work fits twice, and why a
single-station corner frequency should be read as an input to an event
estimate rather than as a measurement in itself.

### The same thing, as one call

Everything above is the workflow written out, and it is written out on purpose
— the two stages and the weighted mean between them are the method, not an
implementation detail, and a reader who has not seen them cannot judge a
result that came out of them.

But nobody should have to retype it. `specmod.staged.fit_event` is those
fifteen lines with every choice defaulted from `[fitting]`, and it should
reproduce the number we just computed by hand exactly. If it does not, one of
the two is wrong.

In [26]:
from specmod.staged import ChannelSelection, fit_event

staged = fit_event(spectra)          # the whole thing, configured defaults
print(staged.describe())
print()
print(f"by hand : {event_fc['powell']:.4f} Hz")
print(f"API     : {staged.value:.4f} Hz")
print(f"agree   : {abs(staged.value / event_fc['powell'] - 1) < 1e-12}")

fc = 11.59 from 28 channels, weighted by inverse_distance
  stage-1 range 2.835 to 59.64 (490% of the event value)

by hand : 11.5851 Hz
API     : 11.5851 Hz
agree   : True


`describe()` prints the spread as well as the mean, and that is deliberate. A
2% spread and a 300% spread give the same weighted mean and mean completely
different things; a corner frequency reported without it is a number with no
error on it.

### Which stations vote is a decision, and a big one

Everything so far has averaged over all 28 channels. That is rarely what you
want after looking at the data. A clipped record, a bad instrument response or
a pick on the wrong phase gives a corner frequency that is *confidently*
wrong, and averaging it in moves the event value for every other station.

Two things make that lever bigger than it first looks. Inverse-distance
weighting is concentrated — on epicentral distance here the nearest two
channels carry over 40% of the total weight — and stress drop goes as $f_c^3$,
so a modest shift in the corner is a large shift in the thing you report.

Which distance measure you choose feeds straight into this, and
`specmod.distance` makes it a registry for that reason: `repi` and `rhyp` are
implemented, and `rrup`/`rjb` are registered but raise, since a point source
has no rupture surface to measure from.

In [27]:
import numpy as np

ids = list(staged.contributing)
distance = np.array([spectra[i].signal.meta[metric] for i in ids])
w = 1 / distance
w = w / w.sum()
order = np.argsort(-w)

print("weight carried by the nearest channels:")
for k in (1, 2, 4, 8):
    print(f"  nearest {k:2d}: {100 * w[order[:k]].sum():5.1f}%")
print()
print(f"the single nearest is {ids[order[0]]} at {distance[order[0]]:.2f} km ({metric})")

weight carried by the nearest channels:
  nearest  1:  20.7%
  nearest  2:  41.4%
  nearest  4:  52.5%
  nearest  8:  68.2%

the single nearest is UR.AQ04.00.HHN at 0.89 km (repi)


In [28]:
# Drop that station. A bare station code matches every channel it has —
# `"HHE"` would match a component, `"UR"` a network, `"UR.AQ04.00.HHE"` one
# channel.
# `nearest`, not `station` — that name is bound to a full trace id further up
# and is read again when the spectra are saved.
nearest = ids[order[0]].split(".")[1]
without = fit_event(spectra, selection=ChannelSelection(exclude=(nearest,)))

print(f"all channels    fc = {staged.value:6.3f} Hz  ({len(staged.contributing)} channels)")
print(f"without {nearest:7s} fc = {without.value:6.3f} Hz  ({len(without.contributing)} channels)")
ratio = without.value / staged.value
print(f"  change in fc            {100 * (ratio - 1):+.1f}%")
print(f"  change in stress drop   {ratio**3:.2f}x")
print()
for id, why in sorted(without.excluded.items()):
    print(f"  {id}: {why}")

all channels    fc = 11.585 Hz  (28 channels)
without AQ04    fc = 15.602 Hz  (26 channels)
  change in fc            +34.7%
  change in stress drop   2.44x

  UR.AQ04.00.HHE: matched exclude='AQ04' at station
  UR.AQ04.00.HHN: matched exclude='AQ04' at station


One quality-control decision, a factor of 1.5 in stress drop. That is not an
argument against making the decision — it is an argument for making it
deliberately, writing it into the study file rather than a notebook cell, and
reporting it. `exclude` lives in `[fitting]` for exactly that reason, and every
exclusion comes back with the reason and the level it matched at.

### One trap worth knowing about

`require_pass` drops a station whose stage-1 fit ended with a parameter pinned
against one of its bounds — the minimiser saying "further, if you would let
me", with the bound reported instead of a measurement. Sensible. But the test
is whether $value \pm \sigma$ reaches the bound, and Powell estimates no
covariance matrix, so $\sigma$ is missing and the test almost never fires.

In [29]:
for method in ("powell", "leastsq"):
    run = fit_event(spectra, method=method)
    print(f"{method:8s} {len(run.contributing):2d} channels vote,  event fc {run.value:6.3f} Hz")

same = ChannelSelection(require_pass=False)
print()
print("with the ensemble held fixed at all 28:")
for method in ("powell", "leastsq"):
    run = fit_event(spectra, method=method, selection=same)
    print(f"{method:8s} {len(run.contributing):2d} channels vote,  event fc {run.value:6.3f} Hz")

powell   28 channels vote,  event fc 11.585 Hz


leastsq  22 channels vote,  event fc  5.407 Hz

with the ensemble held fixed at all 28:


powell   28 channels vote,  event fc 11.585 Hz


leastsq  28 channels vote,  event fc 11.535 Hz


So changing the minimiser changes *which stations vote*, not only how each one
is fitted. Compared naively the two look 144% apart; compared over the same
ensemble they agree to 0.6%. Almost all of that gap is the six stations
`leastsq` rejects and Powell cannot.

Neither setting is wrong. `require_pass=True` is doing the right thing when it
fires. But a study comparing minimisers, or quoting a corner frequency
alongside one obtained another way, has to hold the ensemble fixed or say that
it did not.

## 5. Save the results

Two formats, because the data is used two different ways.

**Spectra go to HDF5** — one file per event, one group per channel, with the
units, duration and sampling rate stored as attributes rather than assumed.
Nothing in the file names a Python class, which is the whole point: the
previous format was pickle, and a pickle stops loading the moment a class is
renamed.

**Fit tables go to Parquet**, which keeps its dtypes and can be queried by
DuckDB or polars without being loaded. CSV is still written when you ask for
it, because journal supplements want one.

In [30]:
from specmod.io import load, save

path = save(OUTPUT / spectra.event, spectra)
print(f"{path}  ({path.stat().st_size // 1024} KB)")

back = load(path)
print(f"reloaded {len(back)} spectra; band unchanged: "
      f"{back[station].band == spectra[station].band}")

Spectra/2019-08-26T07:49:24.200000Z.h5  (391 KB)
reloaded 28 spectra; band unchanged: True


In [31]:
FitSpectra.write_flatfile(OUTPUT / "FlatFiles" / f"{spectra.event}.parquet", fits)
FitSpectra.write_flatfile(OUTPUT / "FlatFiles" / f"{spectra.event}.csv", fits)
sorted(p.name for p in (OUTPUT / "FlatFiles").iterdir())

['2019-08-26T07:49:24.200000Z.csv', '2019-08-26T07:49:24.200000Z.parquet']

---

## Where to go next

- [`docs/processing.md`](../docs/processing.md) — every stage above with its
  equation and a pointer to the code that applies it.
- `specmod.config` — the layered configuration. A study pins its values in a
  committed TOML file; `specmod config show` prints what a run resolved to.
- `specmod.transforms` — the five spectral estimators and the Parseval
  contract they share.
- `specmod.sources` — Brune and Boatwright sources, constant and
  frequency-dependent Q.